In [5]:
from lib import get_model


model = get_model()


In [6]:
from langchain_core.messages import HumanMessage

model.invoke([("user", "안녕")])

AIMessage(content='안녕하세요! 😊  \n무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'model': 'coolsoon/kanana-1.5-8b', 'created_at': '2026-09-21T05:30:16.257172Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5390034250, 'load_duration': 4575205416, 'prompt_eval_count': 17, 'prompt_eval_duration': 177969000, 'eval_count': 16, 'eval_duration': 613890000, 'logprobs': None, 'model_name': 'coolsoon/kanana-1.5-8b', 'model_provider': 'ollama'}, id='lc_run--01a0c271-7621-77e2-a424-6fa7de8b552d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 16, 'total_tokens': 33})

#### Runnable
- LangChain에서 `input → 처리 → output`을 수행하는 실행 단위
- `.invoke()`로 실행 가능
- `|`를 이용해 다른 Runnable과 연결 가능
- Prompt, Model, OutputParser 등이 Runnable로 동작
- Runnable들을 연결하면 하나의 Chain/Pipeline을 만들 수 있음

#### RunnableLambda
- 일반 Python 함수를 LangChain의 `Runnable` 객체로 변환하는 클래스
- 변환 후 `.invoke()` 사용 가능
- `|` 연산자로 다른 Runnable과 연결 가능
- `RunnableLambda(function)` 형태로 사용
- 앞선 chain이 runnable이면 뒤에 있는 함수는 자동으로 runnable

#### Runnable Interface

LangChain의 `Runnable`은 공통적으로 실행할 수 있는 인터페이스를 제공함.

##### 동기 실행
- `invoke(...)`
  - 입력 1개를 넣고 결과 1개를 반환
  - 일반적인 실행 방식

- `stream(...)`
  - 결과를 한 번에 반환하지 않고 생성되는 순서대로 조금씩 반환
  - 반환 단위를 `chunk`라고 함

- `batch([...])`

  - 여러 개의 입력을 한 번에 처리
  - 여러 입력에 대해 `invoke()`를 실행하는 형태

##### 비동기 실행
동기 메서드 앞에 `a`가 붙은 형태.

- `ainvoke(...)`
  - `invoke()`의 비동기 버전

- `astream(...)`
  - `stream()`의 비동기 버전

- `abatch([...])`
  - `batch()`의 비동기 버전

#### 정리

| 동기 | 비동기 | 의미 |
|---|---|---|
| `invoke()` | `ainvoke()` | 입력 1개 실행 |
| `stream()` | `astream()` | 결과를 `chunk` 단위로 순차 반환 |
| `batch()` | `abatch()` | 여러 입력을 한 번에 처리 |

```text
Runnable
├── 동기
│   ├── invoke()
│   ├── stream()
│   └── batch()
│
└── 비동기 ----> coroutine 환경
    ├── ainvoke()
    ├── astream()
    └── abatch()

In [15]:
# RunnableLambda
from langchain_core.runnables import RunnableLambda, chain

@chain
def func_a(message: str) -> str:
    return f"func_a:{message}"

def func_b(message: str) -> str:
    return f"func_b:{message}"

#func_a_runnable = RunnableLambda(func_a)
#func_b_runnable = RunnableLambda(func_b)

In [16]:
chain = func_a_runnable | func_b_runnable  # a는 b의 반환값으로 들어감

chain.invoke("안녕하세요!")

'func_b:func_a:안녕하세요!'

In [ ]:
# func_a_runnable.invoke("안녕하세요!")


'func_a:안녕하세요!'

In [23]:
import re 

def protected_message(message):
    pattern = re.compile(r"(010)\D*(\d{4})\D*(\d{4})")

    message = pattern.sub(r"\g<1>-****-****", message) 
    return message

protected_message("나의 전화번호: 010-9999-9999")


'나의 전화번호: 010-****-****'

In [ ]:
import re
# 정규표현식(Regex)을 사용하기 위한 Python 기본 모듈

from langchain_core.messages import HumanMessage
# 사용자가 입력한 메시지를 나타내는 LangChain 메시지 객체

from langchain_core.runnables import chain
# 일반 Python 함수를 LangChain의 Runnable로 만들어주는 decorator

from langchain_core.output_parsers import StrOutputParser
# AIMessage 형태의 모델 출력을 일반 문자열(str)로 변환

from lib import get_model
# lib.py에서 우리가 만든 모델 생성 함수 가져오기


@chain
# protected_message() 함수를 Runnable 객체처럼 사용할 수 있게 만듦
# 따라서 나중에 | 연산자로 model과 연결 가능
def protected_message(message: str) -> list[HumanMessage]:

    pattern = re.compile(r"(010)\D*(\d{4})\D*(\d{4})")
    # 전화번호 패턴 정의
    # (010)   → 첫 번째 그룹: 010
    # \D*     → 숫자가 아닌 문자가 0개 이상 (-, 공백 등)
    # (\d{4}) → 숫자 4자리
    # \D*     → 숫자가 아닌 문자
    # (\d{4}) → 숫자 4자리
    #
    # 예:
    # 010-1234-5678
    # 010 1234 5678
    # 01012345678

    message = pattern.sub(r"\g<1>-****-****", message)
    # 전화번호를 마스킹
    #
    # 010-1234-5678
    # ↓
    # 010-****-****
    #
    # \g<1> = 첫 번째 캡처 그룹인 "010"

    return [HumanMessage(message)]
    # 문자열을 LangChain HumanMessage 객체로 변환
    # 모델이 받을 수 있도록 list 형태로 반환


model = get_model()
# ChatModel 생성
# 이것도 Runnable 객체


parser = StrOutputParser()
# 모델의 AIMessage 출력을 str로 바꿔주는 Runnable 객체


pipeline = protected_message | model | parser
# Runnable들을 | 로 연결하여 하나의 RunnableSequence 생성
#
# 입력 문자열
#     ↓
# protected_message
#     ↓
# 전화번호 마스킹 + HumanMessage 변환
#     ↓
# model
#     ↓
# AIMessage 생성
#     ↓
# parser
#     ↓
# 일반 문자열(str)

In [27]:
chain.invoke("나의 전화번호는 010-1111-2222")

'알려주신 전화번호 형식은 일반적으로 한국의 휴대폰 번호를 나타내는 표기입니다.  \n예시로 주신 `010-****-****`에서  \n- `010`은 한국의 대표적인 휴대폰 코드입니다.  \n- `****`는 4자리의 숫자(예: 1234)로, 개인 정보 보호를 위해 마스킹 처리되었습니다.  \n- `****`도 4자리의 숫자입니다.  \n- `1234` 부분은 실제 본인의 숫자를 입력해야 합니다.\n\n전화번호를 완전한 형태로 입력하면 예를 들어  \n`010-1234-5678`  \n처럼 됩니다.\n\n전화번호를 입력할 때는  \n- 하이픈(-)이나 공백은 생략하고 `01012345678`처럼 숫자만 입력해도 인식됩니다.\n- 하지만, 전화번호 포맷은 시스템이나 상황에 따라 다를 수 있습니다.\n\n혹시 전화번호를 다른 형식(숫자만, 혹은 다른 포맷)으로 변환하거나,  \n전화번호 관련 작업이 필요하신가요?  \n아니면 단순히 전화번호 예시를 원하신 건가요?  \n추가로 궁금한 점이 있으면 말씀해 주세요!'

In [30]:
for chunk in chain.stream("나의 전화번호는 010-2002-3903입니다. 당신의 번호는?"):
    print(chunk, end="") # token별로 하나씩 출력

죄송합니다. 저는 실제로 전화번호를 저장하거나 받을 수 없습니다.  
만약 전화번호를 안전하게 보관하거나 복구를 위해 메모해 두신 거라면, 다른 안전한 곳에 기록해 두시는 것을 추천드립니다.

혹시 전화번호 관리 방법, 분실 시 대처법, 또는 다른 IT 관련 도움이 필요하시면 언제든 말씀해 주세요!

In [32]:
messages = [
    "나의 전화번호는 010-1010-0000입니다. 당신의 번호는?",
    "안녕하세요!",
    "반갑습니다."
]

for i, res in enumerate(chain.batch(messages)):
    print(i, res)

chain.batch(messages)

0 죄송합니다. 저는 실제로 전화번호를 저장하거나 받을 수 없습니다.  
만약 전화번호를 안전하게 보관하거나 복구를 위해 메모해 두신 거라면, 다른 안전한 곳에 기록해 두시는 것을 추천드립니다.

혹시 전화번호 관리 방법, 분실 시 대처법, 또는 다른 IT 관련 도움이 필요하시면 언제든 말씀해 주세요!
1 안녕하세요! 😊  
오늘도 좋은 하루 보내세요.  
무엇을 도와드릴까요?
2 반갑습니다! 무엇을 도와드릴까요? 😊


['죄송합니다. 저는 실제로 전화번호를 저장하거나 받을 수 없습니다.  \n만약 전화번호를 안전하게 보관하거나 복구를 위해 메모해 두신 거라면, 다른 안전한 곳에 기록해 두시는 것을 추천드립니다.\n\n혹시 전화번호 관리 방법, 분실 시 대처법, 또는 다른 IT 관련 도움이 필요하시면 언제든 말씀해 주세요!',
 '안녕하세요! 😊  \n오늘도 좋은 하루 보내세요.  \n무엇을 도와드릴까요?',
 '반갑습니다! 무엇을 도와드릴까요? 😊']

In [37]:
async def async_messages():
    async for chunk in chain.astream("나의 전화번호는 010-1010-0000입니다. 당신의 번호는?"):
        print(chunk, end="") # token별로 하나씩 출력

In [38]:
await async_messages()

죄송합니다. 저는 실제로 전화번호를 저장하거나 받을 수 없습니다.  
만약 전화번호를 안전하게 보관하거나 복구를 위해 메모해 두신 거라면, 다른 안전한 곳에 기록해 두시는 것을 추천드립니다.

혹시 전화번호 관리 방법, 분실 시 대처법, 또는 다른 IT 관련 도움이 필요하시면 언제든 말씀해 주세요!